### Simulation Study: Analysis by Group Differences vs. Overall Difference in a Cross-over Trial

Brief:
Build a simulation study that implements the Hills & Armitage (1979) crossover trial in code. 
1. Generate simulated data
2. Implement both analysis strategies (calculating group differences vs. overall difference)
3. See if we can replicate the analytical result from the paper in code.

#### Setup

In [1]:
# Importing libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Setting seed value for reproducibility
seed = 929

# Seeding numpy's global generator so np.random draws are reproducible
np.random.seed(seed)


#### Trial Parameters

Using Example 1 from Hills & Armitage (1979) for loose inspiration.

Some of these parameters are defined up-front so we have references or targets to compare our simulations to.

Let $(e_1 - e_2)$ be the time trend for a patient (independent of treatment, the underlying disease might naturally improve or worsen over time). "If there is any consistent time trend, the values of $(e_1 - e_2)$ over all patients will have a non-zero mean of $\delta$ w/ standard deviation $\sigma$" (which we will call sigma_within here to reflect that the variation comes from random biological noise).

Therefore:

$\delta$ = population average time trend, or the average amount the disease naturally changes from Period 1 to Period 2 across all subjects 

$\sigma_{within}$ = variation of individual time trends around $\delta$, reflects the random biological jitter within the same patient from one period to the next (time-varying)

$\mu_{baseline}$ = population's average starting baseline response, i.e, how sick or healthy they are before any treatment is given

$\sigma_{between}$ = how widely individual patients vary around that average baseline (time-invariant)



In [2]:
# Sample sizes
n_A = 17        # Group A (X, Y)
n_B = 12        # Group B (Y, X)

# True fixed treatment effects 
T_x = 5.0                   # fixed treatment effect for Treatment X
T_y = 2.0                   # fixed treatment effect for Treatment Y

# Defining the true relative treatment effect as the difference between the two true parameters
true_treatment_effect = T_x - T_y

# Other parameters

delta = 1.5                 # population average time trend
sigma_within = 3.0          # stdev of individual time trends around mean delta (within-patient 
                            # variability)
                            # epsilon (each individual patient's random noise) gets folded in here

mu_baseline = 50.0          # overall population mean of patient baselines
sigma_between = 10.0        # variation of mu_baseline or how much patients' baselines differ 
                            # across the population 
                            # (between-patient variability)


#### Simulating Trial Data

Patients are randomly allocated to Group A or Group B and assessed during each period. 

- Group A gets Treatment X in Period 1, then Treatment Y in Period 2.
- Group B gets Treatment Y in Period 1, then Treatment X in Period 2. 

In [3]:
def simulate_crossover_data(n_A, n_B, T_x, T_y, delta, sigma_within, mu_baseline, sigma_between):
    """
    A function that generates simulated cross-over trial data based on Hills & Armitage (1979).

    Parameters
    --
    n_A : Number of patients in Group A (X,Y)
    n_B : Number of patients in Group B (Y,X)
    T_x : True fixed treatment effect for Treatment X
    T_y : True fixed treatment effect for Treatment Y
    delta : Population average time trend
    sigma_within : Standard deviation of individual time trends (within-patient variability)
    mu_baseline : Population average starting baseline
    sigma_between : Standard deviation of personal biological baselines (between-patient variability)

    Returns
    --
    pd.DataFrame : A simulated trial dataset

    """

    # Simulating patient-specific biological baselines via random draws from a normal distribution with mean mu_baseline and stdev sigma_between
    baselines_A = np.random.normal(mu_baseline, sigma_between, n_A)
    baselines_B = np.random.normal(mu_baseline, sigma_between, n_B)

    # Simulating time trend (e_1 - e_2), which should have a mean of delta and a stdev of sigma_within (variation within patients)

    # First we start by generating random noise epsilon via random draws from a normal distribution with mean 0 and stdev sigma_within / sqrt(2)

    # div by sqrt(2) bc np.random.normal calls for the SD (not the variance)
    # subtracting two independent noises adds their variances, so we have to make sure each noise gets half the target variance so we add back up properly in the end

    # Group A
    eps_1A = np.random.normal(0, sigma_within / np.sqrt(2), n_A)
    eps_2A = np.random.normal(0, sigma_within / np.sqrt(2), n_A)

    # Group B
    eps_1B = np.random.normal(0, sigma_within / np.sqrt(2), n_B)
    eps_2B = np.random.normal(0, sigma_within / np.sqrt(2), n_B)

    # Now calculating e values as a sum of baseline, delta, and epsilon
    # Keeping delta in just the Period 1 term so it doesn't accidentally cancel out    
    
    # Group A
    e1_A = baselines_A + delta + eps_1A
    e2_A = baselines_A + eps_2A

    # Group B
    e1_B = baselines_B + delta + eps_1B
    e2_B = baselines_B + eps_2B

    # Calculating patient responses y1 and y2

    # Group A (X,Y)
    y1_A = e1_A + T_x
    y2_A = e2_A + T_y

    # Group B (Y,X)
    y1_B = e1_B + T_y
    y2_B = e2_B + T_x

    # Packaging simulated data into nice dataframes
    df_A = pd.DataFrame({
        'Group': 'A', 
        'y1': y1_A, 
        'y2': y2_A
        })
    
    df_B = pd.DataFrame({
        'Group': 'B',
        'y1': y1_B,
        'y2': y2_B
    })

    # Returning the concatenated dataframes
    return pd.concat(
        [df_A, df_B], 
        ignore_index=True       # clean index in new df
        )

# Calling the function to simulate data for a crossover trial with the trial parameters defined above
df_trial = simulate_crossover_data(
    n_A,
    n_B,
    T_x,
    T_y,
    delta, 
    sigma_within,
    mu_baseline,
    sigma_between
)

# Inspecting the simulated data, 10 random points each from Group A and Group B
df_trial.groupby('Group').sample(n=10, random_state=seed)

,Group,y1,y2
6,A,51.105169,46.255922
11,A,62.884883,61.447797
3,A,54.953197,52.338053
2,A,62.602314,55.592975
5,A,40.477997,33.925978
0,A,46.404976,42.221202
8,A,28.190986,24.102469
13,A,68.664425,67.949562
15,A,70.519386,69.371499
4,A,41.879230,40.401413


#### Variance Helper Function

In [4]:
def calculate_standard_error(var_A, var_B, n_A, n_B, method, p_A=1, p_B=1, is_group_diff=False):

    """
    Helper function to calculate standard error via both pooled (t-test) and unpooled (Z-test) methods.

    Parameters
    ---
    var_A : Variance of Group A
    var_B : Variance of Group B
    n_A : Sample size of Group A
    n_B : Sample size of Group B
    method : 'pooled' (for pooled t-test, assuming equal variance) or 'unpooled' (for Z-test, assuming unequal variances)
    p_A (optional): DoF "penalty" for number of parameters estimated in Group A. Defaults to 1 for a group mean.
    p_B (optional): DoF "penalty" for number of parameters estimated in Group B. Defaults to 1 for a group mean.
    is_group_diff (optional): Defaults to False, if True then scales the standard error by 0.5.

    Returns
    ---
    tuple : (se, final_dof) containing the standard error and degrees of freedom 

    """ 

    # Assuming the stdevs estimate the same thing, we can pool them
    if method == 'pooled':

        # Calculating pooled variance weighted by dof penalties
        s2_pooled = (((n_A - p_A) * var_A) + ((n_B - p_B) * var_B)) / (n_A + n_B - (p_A + p_B))

        var_term = (s2_pooled / n_A) + (s2_pooled / n_B)

        final_dof = n_A + n_B - (p_A + p_B)

    elif method == 'unpooled':

        var_term = (var_A / n_A) + (var_B / n_B)

        # Hard-coding infinite dofs to trigger the Z-test
        final_dof = np.inf

    else:
        raise ValueError("Method has to be either 'pooled' or 'unpooled'")

    # H&A scale the group diff estimator by 1/2, so we scale by 1/2 as long as we are analyzing by group differences
    # If is_group_diff = False, we skip the scaling
    if is_group_diff:
        se = 0.5 * np.sqrt(var_term)
    else:
        se = np.sqrt(var_term)

    return se, final_dof


#### Analysis by Group Differences

In [5]:
def group_diff_analysis(df, method, p_A=1, p_B=1):
    """
    Function that analyzes data by group treatment differences.

    Parameters
    --
    df : DataFrame that should have columns 'Group' ('A' or 'B'), 'y1' (Period 1 response), 'y2' (Period 2 response).
    method : 'pooled' (t-test) or 'unpooled' (Z-test). 
    p_A (optional) : dof "penalty" for Group A, passed to calculate_standard_error. Defaults to 1 assuming a group mean has been calculated.
    p_B (optional) : dof "penalty" for Group B, passed to calculate_standard_error. Defaults to 1 assuming a group mean has been calculated.
    
    Returns
    --
    dict : treatment_effect (estimate of T_x - T_y), period_effect (estimate of delta), se, dof, treatment_stat, p_val_treatment, period_stat, p_val_period
    """

    # Defining groups as fresh dfs
    df_A = df[df['Group'] == 'A'].copy()
    df_B = df[df['Group'] == 'B'].copy()

    # Calculating sequence-specific differences 

    # Group A: d_A = y1 - y2
    df_A['d'] = df_A['y1'] - df_A['y2']
    # Group B: d_B = y2 - y1
    df_B['d'] = df_B['y2'] - df_B['y1']

    # Calculating sample stats
    mean_dA = df_A['d'].mean()
    mean_dB = df_B['d'].mean()
    var_dA = df_A['d'].var(ddof=p_A)
    var_dB = df_B['d'].var(ddof=p_B)
    n_A, n_B = len(df_A), len(df_B)

    # Defining estimators 

    # From H&A, 1/2(d_A + d_B) estimates T_x - T_y
    est_treatment_effect = 0.5 * (mean_dA + mean_dB)

    # From H&A, 1/2(d_A - d_B) estimates delta
    est_period_effect = 0.5 * (mean_dA - mean_dB)

    # Calculating standard error and degrees of freedom
    # From H&A, we derive the same SE for both treatment and period effect
    se, dof = calculate_standard_error(
        var_A=var_dA,
        var_B=var_dB,
        n_A=n_A,
        n_B=n_B,
        method=method,
        p_A=p_A,
        p_B=p_B,
        is_group_diff=True
    )

    # Calculating treatment effect test statistic and p-value
    treatment_stat = est_treatment_effect / se

    # stats.t.cdf(x, df=dof) is the probability that a t-variable with dof degrees of freedom is <= x
    # (df is scipy's name for degrees of freedom, not a dataframe)
    p_val_treatment = 2 * (     # 2* makes this a two-tailed test
        1 - stats.t.cdf(        # 1-cdf is the area in the upper tail beyond our statistic
        abs(treatment_stat),    # makes the formula work regardless of the sign
        df=dof)                 # a t-dist with infinite dofs is the standard normal, so this function works for both finite and infinite dofs
        )

    # Calculating period effect test statistic and p-value (same process as above)
    period_stat = est_period_effect / se
    p_val_period = 2 * (1 - stats.t.cdf(abs(period_stat), df=dof))    

    return {
        'treatment_effect': est_treatment_effect,
        'period_effect': est_period_effect,
        'se': se,
        'dof': dof,
        'treatment_stat': treatment_stat,
        'p_val_treatment': p_val_treatment,
        'period_stat': period_stat,
        'p_val_period': p_val_period
    }

#### Analysis by Overall Difference

In [6]:
def overall_diff_analysis(df, method):
    """
    Function that analyzes data by overall difference d_bar, ignoring order/sequence.

    Parameters
    ---
    df : DataFrame that should have columns 'Group' ('A' or 'B'), 'y1' (Period 1 response), 'y2' (Period 2 response).
    method : 't' (t-test on N-1 dof) or 'Z' (Z-test / normal deviate).


    Returns
    ---
    dict : treatment_effect (estimate of T_x - T_y), sd, se, dof, treatment_stat, p_val_treatment

    """

    # Splitting passed df into Groups A and B
    df_A = df[df['Group'] == 'A']
    df_B = df[df['Group'] == 'B']

    # Same d_A and d_B as the group diff analysis

    # Group A: d_A = y1 - y2
    d_A = df_A['y1'] - df_A['y2']

    # Group B: d_B = y2 - y1
    d_B = df_B['y2'] - df_B['y1']

    # Ignoring order by pooling all differences into one sample
    d = pd.concat([d_A, d_B]) 
    N = len(d)

    # We can derive that d_bar estimates T_x - T_y (but is unbiased only when patients are allocated completely at random)
    d_bar = d.mean()
    est_treatment_effect = d_bar

    # And we can also derive that SD(d) estimates sqrt(sigma^2 + delta^2), which is the "penalty in accuracy" that H&A reference
    sd = d.std(ddof=1)      # using dof=1 because we estimated one mean

    # Calculating standard error
    se = sd / np.sqrt(N)

    # Defining dof

    # t-test on N - 1 dof (one sample, one mean estimated)
    if method == 't':
        dof = N - 1

    # hard-coding infinite dofs to force the Z-test
    elif method == 'Z':
        dof = np.inf

    else:
        raise ValueError("Method has to be either 't' or 'Z'")


    # Calculating test statistic and p-value
    treatment_stat = est_treatment_effect / se

    p_val_treatment = 2 * (1-stats.t.cdf(abs(treatment_stat), df=dof))

    return {
        'treatment_effect': est_treatment_effect,
        'sd': sd,
        'se': se,
        'dof': dof,
        'treatment_stat': treatment_stat,
        'p_val_treatment': p_val_treatment
    }
    

#### Single Trial Check

Just checking to make sure that all seems to be in working order before we run simulations



In [ ]:
print("Single Trial Results")

# Printing true values for reference 
print(f"True Treatment Effect: {true_treatment_effect:.2f} | True Period Effect: {delta:.2f}\n")

# Group Diff, Pooled t-Test
gd_p = group_diff_analysis(df_trial, method='pooled')

print(f"Group Diff (Pooled t-test):   Treatment Est={gd_p['treatment_effect']:.3f}, Period Est={gd_p['period_effect']:.3f}, SE={gd_p['se']:.3f}, dof={gd_p['dof']:.1f}, p={gd_p['p_val_treatment']:.4f}")


# Group Diff, Unpooled Z-test
gd_u = group_diff_analysis(df_trial, method='unpooled')

print(f"Group Diff (Unpooled Z-test): Treatment Est={gd_u['treatment_effect']:.3f}, Period Est={gd_u['period_effect']:.3f}, SE={gd_u['se']:.3f}, dof={gd_u['dof']}, p={gd_u['p_val_treatment']:.4f}")


# Overall Diff, t-test
od_t = overall_diff_analysis(df_trial, method='t')

print(f"Overall Diff (t-test):  Treatment Est={od_t['treatment_effect']:.3f}, Period Est=N/A, SD={od_t['sd']:.3f}, SE={od_t['se']:.3f}, dof={od_t['dof']}, p={od_t['p_val_treatment']:.4f}")


# Overall Diff, Z-test
od_z = overall_diff_analysis(df_trial, method='Z')

print(f"Overall Diff (Z-test):  Treatment Est={od_z['treatment_effect']:.3f}, Period Est=N/A, SD={od_z['sd']:.3f}, SE={od_z['se']:.3f}, dof={od_z['dof']}, p={od_z['p_val_treatment']:.4f}")


Single Trial Results
True Treatment Effect: 3.00 | True Period Effect: 1.50

Group Diff (Pooled t-test):   Treatment Est=2.387, Period Est=0.454, SE=0.478, dof=27.0, p=0.0000
Group Diff (Unpooled Z-test): Treatment Est=2.387, Period Est=0.454, SE=0.488, dof=inf, p=0.0000
Overall Diff (t-test):  Treatment Est=2.465, Period Est=N/A, SD=2.532, SE=0.470, dof=28, p=0.0000
Overall Diff (Z-test):  Treatment Est=2.465, Period Est=N/A, SD=2.532, SE=0.470, dof=inf, p=0.0000


#### Running Simulations

In [ ]:
# Setting number of simulations to run
n_sims = 10000

# Empty tupperware for metrics
results = {
    'gd_pooled_treatment': [], 'gd_pooled_period': [], 'gd_pooled_se': [],
    'gd_unpooled_treatment': [], 'gd_unpooled_period': [], 'gd_unpooled_se': [],
    'od_treatment': [], 'od_sd': [], 'od_se': []
}

for _ in range (n_sims):
    sim_df = simulate_crossover_data(n_A, n_B, T_x, T_y, delta, sigma_within, mu_baseline, sigma_between)

    # Simulating group diff analyses
    res_gd_p = group_diff_analysis(sim_df, method='pooled')
    res_gd_u = group_diff_analysis(sim_df, method='unpooled')

    # Simulating overall diff analyses
    # (Estimate, SD and SE are the same for 't' and 'Z' and only dof differs, so we just call it once here)
    res_od = overall_diff_analysis(sim_df, method='t')

    # Storing outputs

    # Group diffs, pooled
    results['gd_pooled_treatment'].append(res_gd_p['treatment_effect'])
    results['gd_pooled_period'].append(res_gd_p['period_effect'])
    results['gd_pooled_se'].append(res_gd_p['se'])

    # Group diffs, unpooled
    results['gd_unpooled_treatment'].append(res_gd_u['treatment_effect'])
    results['gd_unpooled_period'].append(res_gd_u['period_effect'])
    results['gd_unpooled_se'].append(res_gd_u['se'])

    # Overall diff
    results['od_treatment'].append(res_od['treatment_effect'])
    results['od_sd'].append(res_od['sd'])
    results['od_se'].append(res_od['se'])


print(f"SIMULATION RESULTS")

print("TARGETS")
print(f"Treatment effect (T_x - T_y): {true_treatment_effect:.4f}")
print(f"Period effect (delta):        {delta:.4f}\n")

print("TREATMENT EFFECT ESTIMATE")
print(f"Group diff, pooled:     {np.mean(results['gd_pooled_treatment']):.4f}")
print(f"Group diff, unpooled:   {np.mean(results['gd_unpooled_treatment']):.4f}")
print(f"Overall diff:           {np.mean(results['od_treatment']):.4f}\n")

print("PERIOD EFFECT ESTIMATE")
print(f"Group diff, pooled:     {np.mean(results['gd_pooled_period']):.4f}")
print(f"Group diff, unpooled:   {np.mean(results['gd_unpooled_period']):.4f}")
print(f"Overall diff:           N/A\n")

print("STANDARD ERROR (of the treatment estimate)")
print(f"Group diff, pooled:     {np.mean(results['gd_pooled_se']):.4f}")
print(f"Group diff, unpooled:   {np.mean(results['gd_unpooled_se']):.4f}")
print(f"Overall diff:           {np.mean(results['od_se']):.4f}")



SIMULATION RESULTS (mean over 10000 simulated trials)

TARGETS
Treatment effect (T_x - T_y): 3.0000
Period effect (delta):        1.5000

TREATMENT EFFECT ESTIMATE
Group diff, pooled:     3.0055
Group diff, unpooled:   3.0055
Overall diff:           3.2636

PERIOD EFFECT ESTIMATE
Group diff, pooled:     1.4969
Group diff, unpooled:   1.4969
Overall diff:           N/A

STANDARD ERROR (of the treatment estimate)
Group diff, pooled:     0.5615
Group diff, unpooled:   0.5609
Overall diff:           0.6186
